<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 8 — Python Toolkit</div>
</div>

#### scikit-learn: training and test sets, pipelines, cross-validation and grouped data, trees, forests and nearest neighbours, precision and recall, regression, and feature importance.
***

## Why this notebook

The scikit-learn you need for today's lab and for the data-driven half of Exercise 3. Every model in scikit-learn works the same way (`fit`, then `predict`), so once you have the pattern, swapping a tree for a forest or a neural network is one line. Twenty minutes; skim what you know. Section 8 has optional practice with numbers to check against; a worked copy is next to this notebook.

1. splitting data into training and test sets;
2. pipelines: standardising inside the model;
3. cross-validation, and holding out whole groups;
4. classifiers: a tree, a forest, nearest neighbours;
5. judging a classifier: the confusion matrix, precision and recall;
6. regression: predicting a number, and what happens outside the training range;
7. which features matter: permutation importance.

If arrays, loops or functions feel shaky, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises them first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(8)

## <span style="color:#B5121B">1. Training and test sets</span>

A model that has seen a data point will recognise it, so judge it only on data it has **not** seen. `train_test_split` shuffles and splits; `stratify=y` keeps each class's share the same in both halves; `random_state` makes the split reproducible. Features go in a table `X` (one row per object, one column per feature) and the answer in `y`.

In [ ]:
# two classes in two features: class 1 sits a little up and to the right
n = 400
X = np.vstack([rng.normal(0, 1, (n, 2)), rng.normal(1.2, 1, (n, 2))])
y = np.repeat([0, 1], n)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(X_train.shape, X_test.shape, np.bincount(y_test))

## <span style="color:#B5121B">2. Pipelines: standardise inside the model</span>

Methods that measure distances (nearest neighbours, neural networks) are dominated by whichever feature has the biggest numbers: measure one feature in millimetres and it swamps the rest (Lab 6). `make_pipeline(StandardScaler(), model)` standardises with the *training* data's means and spreads, then applies the same transformation to anything you predict. Trees and forests split one feature at a time, so they don't need it.

In [ ]:
X_mm = X_train * [1, 1000]                   # the same data, second feature in "millimetres"
X_mm_test = X_test * [1, 1000]
raw = KNeighborsClassifier(15).fit(X_mm, y_train)
piped = make_pipeline(StandardScaler(), KNeighborsClassifier(15)).fit(X_mm, y_train)
print(f"k-NN, raw units: {raw.score(X_mm_test, y_test):.3f};  standardised in a pipeline: {piped.score(X_mm_test, y_test):.3f}")

In [ ]:
# Your turn: why does the raw version do worse? Which feature is it effectively using?

## <span style="color:#B5121B">3. Cross-validation, and holding out whole groups</span>

One test set gives one number, which itself varies. **k-fold cross-validation** splits the training data into k parts, trains on k − 1 and tests on the remaining one, k times; the spread of the k scores says how much to trust the mean. Use it to *choose* settings (a tree's depth, the k of k-NN), and keep the test set for the final verdict.

**Holding out groups.** If the data come in groups (several values from one station, one person, one region), a random split puts members of the same group in both training and test, and the model can recognise the group instead of learning the physics. `GroupKFold` keeps each group entirely in one fold, which asks the honest question: *how well does it do on a group it has never seen?*

In [ ]:
# 20 "stations", 30 values each: y depends on x, plus an offset that is particular to each station
st = np.repeat(np.arange(20), 30)
offset = rng.normal(0, 2, 20)[st]
lat = rng.uniform(50, 58, 20)[st]                  # a feature that identifies the station, but does not cause y
x = rng.uniform(0, 10, len(st))
yy = 0.5 * x + offset + rng.normal(0, 0.5, len(st))
F = np.column_stack([x, lat])
model = RandomForestRegressor(200, random_state=0)
for name, cv, groups in [("random 5-fold", KFold(5, shuffle=True, random_state=0), None), ("whole stations held out", GroupKFold(5), st)]:
    s = -cross_val_score(model, F, yy, cv=cv, groups=groups, scoring="neg_root_mean_squared_error")
    print(f"{name:24s} RMSE {s.mean():.2f} ± {s.std():.2f}")

The random split looks far better because the forest learns each station's offset from `lat`: it has seen that station before. On a new station it cannot know the offset. If your question is about places, people or areas you have not measured, hold out whole groups.

## <span style="color:#B5121B">4. Classifiers: a tree, a forest, nearest neighbours</span>

- **Decision tree:** a sequence of yes/no questions on one feature at a time (Lecture 10). Deep trees fit the training data perfectly and new data badly; `max_depth` limits them.
- **Random forest:** many deep trees, each trained on a bootstrap resample of the data (Lecture 5) and a random subset of features at each split; they vote. Averaging many noisy trees cancels much of their noise.
- **k-nearest neighbours:** a new point takes the most common class among its k closest training points. Small k follows noise; large k blurs boundaries.

All three: `model.fit(X_train, y_train)`, `model.predict(X_test)`, `model.score(X_test, y_test)` (accuracy), and `model.predict_proba(X_test)` (the fraction of trees or neighbours voting for each class).

In [ ]:
models = {"tree, depth 3": DecisionTreeClassifier(max_depth=3, random_state=0),
          "tree, no limit": DecisionTreeClassifier(random_state=0),
          "forest, 300 trees": RandomForestClassifier(300, random_state=0),
          "k-NN, k = 15": make_pipeline(StandardScaler(), KNeighborsClassifier(15))}
for name, m in models.items():
    m.fit(X_train, y_train)
    cv = cross_val_score(m, X_train, y_train, cv=5)
    print(f"{name:18s} train {m.score(X_train, y_train):.3f}  cross-validated {cv.mean():.3f} ± {cv.std():.3f}  test {m.score(X_test, y_test):.3f}")

Two things to notice. The unlimited tree and the forest score 1.000 on the training data, which says nothing: only the cross-validated and test numbers count. And the forest is *not* the winner here: two overlapping blobs have a simple boundary, which a shallow tree or k-NN finds, while the forest's deep trees chase the overlap. No method is best everywhere; that is why you cross-validate.

In [ ]:
# Your turn: cross-validate k-NN for k = 1, 5, 15, 50 and 150. Which k would you choose, and why not k = 1?

## <span style="color:#B5121B">5. Judging a classifier: precision and recall</span>

Accuracy hides rare classes: if 95% of objects are class 0, "always 0" scores 95%. For each class, **recall** is the fraction of its members that were found; **precision** is the fraction of those labelled with it that really belong to it. `confusion_matrix` shows where the errors go (rows: true class; columns: predicted); `classification_report` prints precision and recall for every class. `class_weight="balanced"` makes errors on a rare class cost more.

In [ ]:
# a rare class: 5% of objects
Xr = np.vstack([rng.normal(0, 1, (950, 2)), rng.normal(1.8, 1, (50, 2))]); yr = np.repeat([0, 1], [950, 50])
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.5, stratify=yr, random_state=0)
for kw in ({}, {"class_weight": "balanced"}):
    m = RandomForestClassifier(300, min_samples_leaf=5, random_state=0, **kw).fit(Xr_tr, yr_tr)
    p = m.predict(Xr_te)
    print(kw or "no weighting", "\n", confusion_matrix(yr_te, p))
    print(classification_report(yr_te, p, digits=3))

Weighting finds more of the rare class (higher recall) at the cost of more false alarms (lower precision). Which matters more depends on the question: missing a quasar, or wasting telescope time on a star?

## <span style="color:#B5121B">6. Regression: predicting a number</span>

The same models predict numbers: `RandomForestRegressor`, `KNeighborsRegressor`, and the straight line `LinearRegression`. Judge them by the root-mean-square error on unseen data, in the units of what you predict. **Outside the training range**, trees and neighbours cannot follow a trend: a forest predicts an average of training values, so it levels off; a straight line continues (rightly or wrongly). A physical model can extrapolate if its physics holds there.

In [ ]:
xs = rng.uniform(0, 10, 300); ys = 2 * xs + rng.normal(0, 1, 300)
inside = xs < 6                                            # train only on x < 6
regs = {"straight line": LinearRegression(), "forest": RandomForestRegressor(300, random_state=0),
        "k-NN, k = 10": KNeighborsRegressor(10)}
grid = np.linspace(0, 10, 200)[:, None]
plt.figure(figsize=(7, 4)); plt.plot(xs, ys, ".", color="#BEC0C2", ms=3, label="data")
for (name, m), c in zip(regs.items(), ["#1F5FA8", "#B5121B", "#38614A"]):
    m.fit(xs[inside, None], ys[inside])
    out = ~inside
    rmse_in = np.sqrt(np.mean((ys[inside] - m.predict(xs[inside, None]))**2))
    rmse_out = np.sqrt(np.mean((ys[out] - m.predict(xs[out, None]))**2))
    print(f"{name:14s} RMSE on the training data (x < 6) {rmse_in:.2f}, beyond it (x > 6) {rmse_out:.2f}")
    plt.plot(grid, m.predict(grid), color=c, label=name)
plt.axvline(6, color="k", ls=":"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

The forest's small training error is memorisation, not skill. Beyond x = 6 the straight line keeps its error; the forest and k-NN flatten out at the largest values they were trained on.

## <span style="color:#B5121B">7. Which features matter?</span>

**Permutation importance:** shuffle one feature in the test set and see how much the score drops. A feature the model relies on matters a lot; one it ignores, not at all. It works for any model. (Forests also have `feature_importances_`, which can favour features with many distinct values.) Importance says what the model uses, not what causes what.

In [ ]:
m = RandomForestClassifier(300, random_state=0).fit(X_train, y_train)
imp = permutation_importance(m, X_test, y_test, n_repeats=20, random_state=0)
for name, mean, sd in zip(["feature 0", "feature 1"], imp.importances_mean, imp.importances_std):
    print(f"{name}: accuracy drops by {mean:.3f} ± {sd:.3f} when shuffled")

## <span style="color:#FD9029">8. Optional practice</span>

Three short problems. Check your numbers against the **Check** lines.

**P1. Overfitting a tree.** For the data of Section 1, find the training accuracy and the cross-validated accuracy of a tree for `max_depth` = 1, 2, 3, 5, 10 and no limit. Which depth would you choose?

In [ ]:
# P1

**Check:** training accuracy climbs to 1.000 with no limit, while the cross-validated accuracy peaks at a small depth (about 2–3) and then falls: the deep tree is learning the noise.

**P2. A forest is an average of trees.** Fit `RandomForestClassifier` with 1, 10, 100 and 300 trees to the data of Section 1, and compare the test accuracy. Why does it level off?

In [ ]:
# P2

**Check:** one tree does noticeably worse than a hundred; beyond about a hundred trees the accuracy hardly changes.

**P3. The groups again.** In Section 3, drop `lat` from the features (use only `x`) and repeat both kinds of cross-validation. What happens to the gap, and why?

In [ ]:
# P3

**Check:** the two errors become almost equal, and both close to the station-held-out value of Section 3: the gap was the model recognising stations.

## Summary

- **Split** into training and test data; judge only on unseen data; `stratify` for classes, `random_state` for reproducibility.
- **Pipelines** standardise inside the model: essential for k-NN and neural networks, not needed for trees.
- **Cross-validate** to choose settings; **hold out whole groups** (`GroupKFold`) when the question is about new stations, people or regions.
- **Trees** are readable but overfit; **forests** average many trees; **k-NN** depends on k and on scaling.
- **Precision and recall** per class, not accuracy alone, when a class is rare.
- **Regression:** RMSE on unseen data; forests and k-NN cannot extrapolate beyond the training range.
- **Permutation importance** shows what a model uses, not what causes what.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>